In [ ]:
# ============================================
# 🏃 HEALTH & FITNESS METRICS FROM VIDEO
# Upload your MP4 video and get detailed health metrics
# ============================================

# @title **STEP 1: Install Required Libraries**
!pip install ultralytics opencv-python supervision filterpy mediapipe pandas matplotlib scikit-learn scipy -q
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118 -q

import torch
print(f"✅ GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


import torch
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

✅ GPU Available: True
✅ GPU: Tesla T4
✅ GPU Memory: 15.8 GB
✅ GPU: Tesla T4


In [ ]:
# @title **STEP 2: Upload Your MP4 Video**
from google.colab import files
import os

# Create directories
os.makedirs('uploaded_videos', exist_ok=True)
os.makedirs('output_results', exist_ok=True)

print("📤 UPLOAD YOUR MP4 VIDEO FILE")
print("=" * 50)
print("Click 'Choose Files' below to upload your exercise video")
print("Supported: MP4, AVI, MOV, MKV formats")

uploaded = files.upload()

# Get the uploaded file path
video_path = None
for filename in uploaded.keys():
    # Move to organized folder
    new_path = f"uploaded_videos/{filename}"
    os.rename(filename, new_path)
    video_path = new_path
    file_size = os.path.getsize(video_path) / (1024*1024)  # MB
    print(f"\n✅ Upload Successful!")
    print(f"   File: {filename}")
    print(f"   Size: {file_size:.1f} MB")
    print(f"   Path: {video_path}")
    break

if not video_path:
    print("\n⚠️ No video uploaded. Using sample video instead.")
    !wget -q -O uploaded_videos/sample_exercise.mp4 "https://github.com/ultralytics/assets/releases/download/v8.2.0/running.mp4"
    video_path = "uploaded_videos/sample_exercise.mp4"
    print(f"📹 Using sample video: {video_path}")

print("\n" + "=" * 50)

📤 UPLOAD YOUR MP4 VIDEO FILE
Click 'Choose Files' below to upload your exercise video
Supported: MP4, AVI, MOV, MKV formats


Saving Treadmill.mp4 to Treadmill.mp4

✅ Upload Successful!
   File: Treadmill.mp4
   Size: 2.6 MB
   Path: uploaded_videos/Treadmill.mp4



In [ ]:
# @title **STEP 3: Video Information**
import cv2

cap = cv2.VideoCapture(video_path)
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / fps if fps > 0 else 0

cap.release()

print("📹 VIDEO INFORMATION")
print("=" * 50)
print(f"Resolution: {width} x {height}")
print(f"FPS: {fps}")
print(f"Total Frames: {total_frames}")
print(f"Duration: {duration:.1f} seconds")
print(f"Path: {video_path}")

📹 VIDEO INFORMATION
Resolution: 720 x 1280
FPS: 25
Total Frames: 288
Duration: 11.5 seconds
Path: uploaded_videos/Treadmill.mp4


In [ ]:
# @title **STEP 4: Health Metrics Detector Class**
import numpy as np
from collections import deque
import time
import json
from scipy import signal

class FitnessMetricsAnalyzer:
    """
    Analyzes exercise videos for health and fitness metrics
    No calibration needed - uses relative measurements
    """
    def __init__(self, fps=30):
        self.fps = fps
        self.frame_time = 1.0 / fps

        # Store person data
        self.persons = {}
        self.metrics_history = {}

        # Constants for fitness tracking
        self.WALKING_THRESHOLD = 1.4  # m/s (5 km/h)
        self.JOGGING_THRESHOLD = 2.8  # m/s (10 km/h)
        self.RUNNING_THRESHOLD = 4.2  # m/s (15 km/h)

    def register_person(self, person_id, initial_height_pixels):
        """Register a new person for tracking"""
        self.persons[person_id] = {
            'height_pixels': initial_height_pixels,
            'first_seen': time.time(),
            'last_seen': time.time(),
            'frame_count': 0,
            'stride_count': 0,
            'distance_pixels': 0
        }

        # Initialize metrics history
        self.metrics_history[person_id] = {
            'speeds': deque(maxlen=100),  # Store last 100 speed readings
            'positions': deque(maxlen=50),  # Store last 50 positions
            'timestamps': deque(maxlen=50),
            'cadence': deque(maxlen=30),  # Steps per minute
            'intensities': deque(maxlen=100)
        }

        print(f"👤 Registered Person {person_id} (Height: {initial_height_pixels:.0f} pixels)")

    def calculate_pixel_speed(self, person_id, current_position, timestamp):
        """Calculate speed in pixels per second"""
        history = self.metrics_history[person_id]

        # Add current data
        history['positions'].append(current_position)
        history['timestamps'].append(timestamp)

        # Need at least 2 points for speed calculation
        if len(history['positions']) >= 2:
            positions = list(history['positions'])
            timestamps = list(history['timestamps'])

            # Calculate displacement over last N points
            window_size = min(5, len(positions))
            recent_positions = positions[-window_size:]
            recent_times = timestamps[-window_size:]

            total_distance = 0
            total_time = 0

            for i in range(1, len(recent_positions)):
                pos1 = recent_positions[i-1]
                pos2 = recent_positions[i]
                time1 = recent_times[i-1]
                time2 = recent_times[i]

                # Euclidean distance
                distance = np.sqrt((pos2[0]-pos1[0])**2 + (pos2[1]-pos1[1])**2)
                time_diff = time2 - time1

                total_distance += distance
                total_time += time_diff

            if total_time > 0:
                speed_pps = total_distance / total_time  # Pixels per second
                history['speeds'].append(speed_pps)

                # Return smoothed speed (moving average)
                if history['speeds']:
                    return np.mean(list(history['speeds']))

        return 0.0

    def estimate_real_speed(self, pixel_speed, person_height_pixels):
        """
        Estimate real speed in km/h using height as reference
        Assumes average person height of 1.7m
        """
        if person_height_pixels > 0:
            # Convert pixels to meters
            pixels_per_meter = person_height_pixels / 1.7  # Average height
            speed_mps = pixel_speed / pixels_per_meter
            speed_kmh = speed_mps * 3.6
            return speed_kmh
        return 0.0

    def detect_stride(self, person_id, foot_positions):
        """Detect strides/steps from foot movement"""
        history = self.metrics_history[person_id]

        if len(foot_positions) >= 3:
            # Simple peak detection for stride pattern
            y_positions = [pos[1] for pos in foot_positions]  # Vertical movement

            if len(y_positions) > 10:
                # Find peaks (foot strikes)
                peaks, _ = signal.find_peaks(y_positions,
                                           distance=5,  # Minimum frames between steps
                                           prominence=5)  # Minimum prominence

                if len(peaks) > 0:
                    # Update stride count
                    self.persons[person_id]['stride_count'] += len(peaks)

                    # Calculate cadence (steps per minute)
                    if len(history['timestamps']) >= 2:
                        time_window = history['timestamps'][-1] - history['timestamps'][0]
                        if time_window > 0:
                            steps_per_second = len(peaks) / time_window
                            cadence_spm = steps_per_second * 60
                            history['cadence'].append(cadence_spm)

                            return {
                                'stride_count': len(peaks),
                                'cadence_spm': cadence_spm
                            }

        return None

    def classify_intensity(self, speed_kmh):
        """Classify exercise intensity based on speed"""
        if speed_kmh < 3:
            return "Stationary/Stretching", "Very Light"
        elif speed_kmh < 6:
            return "Walking", "Light"
        elif speed_kmh < 10:
            return "Brisk Walking", "Moderate"
        elif speed_kmh < 14:
            return "Jogging", "Vigorous"
        elif speed_kmh < 18:
            return "Running", "High"
        else:
            return "Sprinting", "Maximum"

    def estimate_calories(self, person_id, duration_seconds, avg_speed_kmh):
        """Rough calorie estimation based on MET values"""
        # MET (Metabolic Equivalent of Task) values
        if avg_speed_kmh < 3:
            met = 2.0  # Light stretching
        elif avg_speed_kmh < 6:
            met = 3.5  # Walking
        elif avg_speed_kmh < 10:
            met = 5.0  # Brisk walking
        elif avg_speed_kmh < 14:
            met = 8.0  # Jogging
        elif avg_speed_kmh < 18:
            met = 10.0  # Running
        else:
            met = 16.0  # Sprinting

        # Assume average weight of 70kg
        weight_kg = 70
        calories = met * weight_kg * (duration_seconds / 3600)

        return calories

    def get_summary(self, person_id):
        """Get comprehensive summary for a person"""
        if person_id not in self.metrics_history:
            return None

        history = self.metrics_history[person_id]
        person_data = self.persons.get(person_id, {})

        if not history['speeds']:
            return None

        # Calculate averages
        speeds = list(history['speeds'])
        avg_pixel_speed = np.mean(speeds)
        avg_speed_kmh = self.estimate_real_speed(avg_pixel_speed,
                                               person_data.get('height_pixels', 0))

        # Duration
        if history['timestamps']:
            duration = history['timestamps'][-1] - history['timestamps'][0]
        else:
            duration = 0

        # Cadence
        avg_cadence = np.mean(list(history['cadence'])) if history['cadence'] else 0

        # Intensity classification
        activity, intensity = self.classify_intensity(avg_speed_kmh)

        # Calories
        calories = self.estimate_calories(person_id, duration, avg_speed_kmh)

        # Distance estimate (rough)
        if avg_speed_kmh > 0 and duration > 0:
            distance_km = (avg_speed_kmh * duration) / 3600
        else:
            distance_km = 0

        summary = {
            'person_id': person_id,
            'duration_seconds': duration,
            'avg_speed_kmh': avg_speed_kmh,
            'avg_cadence_spm': avg_cadence,
            'stride_count': person_data.get('stride_count', 0),
            'primary_activity': activity,
            'intensity_level': intensity,
            'estimated_calories': calories,
            'estimated_distance_km': distance_km,
            'avg_pixel_speed': avg_pixel_speed,
            'height_pixels': person_data.get('height_pixels', 0)
        }

        return summary


In [ ]:
# @title **STEP 5: Main Processing Function**
def analyze_exercise_video(input_video_path, output_video=True):
    """
    Main function to process video and extract fitness metrics
    """
    import cv2
    from ultralytics import YOLO
    import supervision as sv
    from tqdm import tqdm

    print(f"\n🚀 STARTING FITNESS ANALYSIS")
    print("=" * 60)
    print(f"Input: {input_video_path}")

    # Open video
    cap = cv2.VideoCapture(input_video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Initialize analyzer
    analyzer = FitnessMetricsAnalyzer(fps=fps)

    # Load YOLOv8 model (pose estimation for better accuracy)
    print("📦 Loading YOLOv8 pose estimation model...")
    model = YOLO('yolov8m-pose.pt')  # Medium model with pose estimation

    # Video writer for output
    if output_video:
        output_path = f"output_results/analyzed_{os.path.basename(input_video_path)}"
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Tracking variables
    next_person_id = 1
    active_tracks = {}
    all_metrics = []

    # Progress bar
    pbar = tqdm(total=total_frames, desc="Processing video")

    print("\n▶️ Processing frames...")

    frame_count = 0
    processing_start = time.time()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        current_time = time.time()

        # Run YOLOv8 pose estimation
        results = model(frame, classes=[0], conf=0.5, verbose=False)[0]

        # Process each detected person
        current_detections = {}

        for i, box in enumerate(results.boxes):
            # Get bounding box
            x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
            confidence = box.conf[0].cpu().numpy()

            # Calculate person height in pixels (from bounding box)
            person_height = y2 - y1

            # Get keypoints if available (for pose estimation)
            if hasattr(box, 'keypoints') and box.keypoints is not None:
                keypoints = box.keypoints[0].cpu().numpy()
                # Use keypoints for better center estimation
                if len(keypoints) > 0:
                    valid_points = keypoints[keypoints[:, 2] > 0.5]  # Visibility > 0.5
                    if len(valid_points) > 0:
                        center_x = np.mean(valid_points[:, 0])
                        center_y = np.mean(valid_points[:, 1])
                    else:
                        center_x, center_y = (x1 + x2) / 2, (y1 + y2) / 2
                else:
                    center_x, center_y = (x1 + x2) / 2, (y1 + y2) / 2
            else:
                center_x, center_y = (x1 + x2) / 2, (y1 + y2) / 2

            # Track person
            person_id = None
            min_distance = 100  # pixels

            for track_id, track_data in active_tracks.items():
                last_center = track_data['last_center']
                distance = np.sqrt((center_x - last_center[0])**2 +
                                 (center_y - last_center[1])**2)

                if distance < min_distance:
                    min_distance = distance
                    person_id = track_id

            # New person if no match found
            if person_id is None:
                person_id = next_person_id
                next_person_id += 1
                analyzer.register_person(person_id, person_height)

            # Update active tracks
            active_tracks[person_id] = {
                'last_center': (center_x, center_y),
                'last_bbox': (x1, y1, x2, y2),
                'last_seen': frame_count,
                'confidence': confidence,
                'height': person_height
            }

            # Calculate metrics
            pixel_speed = analyzer.calculate_pixel_speed(person_id,
                                                        (center_x, center_y),
                                                        current_time)

            speed_kmh = analyzer.estimate_real_speed(pixel_speed, person_height)
            activity, intensity = analyzer.classify_intensity(speed_kmh)

            # Store metrics
            metrics = {
                'timestamp': current_time,
                'frame': frame_count,
                'person_id': person_id,
                'pixel_speed': pixel_speed,
                'speed_kmh': speed_kmh,
                'activity': activity,
                'intensity': intensity,
                'bbox': [x1, y1, x2, y2],
                'center': [center_x, center_y]
            }
            all_metrics.append(metrics)

            # Annotate frame
            # Draw bounding box
            color = (0, 255, 0) if intensity in ["Light", "Moderate"] else (0, 165, 255)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            # Display metrics
            text_y = y1 - 10
            cv2.putText(frame, f"ID:{person_id} {activity}",
                       (x1, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            cv2.putText(frame, f"Speed: {speed_kmh:.1f} km/h",
                       (x1, text_y + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
            cv2.putText(frame, f"Intensity: {intensity}",
                       (x1, text_y + 45), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

        # Clean up old tracks (not seen for 30 frames)
        active_tracks = {tid: data for tid, data in active_tracks.items()
                        if frame_count - data['last_seen'] < 30}

        # Add frame counter
        cv2.putText(frame, f"Frame: {frame_count}/{total_frames}",
                   (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        cv2.putText(frame, f"Active Persons: {len(active_tracks)}",
                   (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

        # Write to output video
        if output_video:
            out.write(frame)

        # Update progress
        pbar.update(1)
        if frame_count % 100 == 0:
            current_fps = frame_count / (time.time() - processing_start)
            pbar.set_postfix({
                'FPS': f'{current_fps:.1f}',
                'Persons': len(active_tracks)
            })

        # Show preview every 200 frames
        if frame_count % 200 == 0:
            from google.colab.patches import cv2_imshow
            preview = cv2.resize(frame, (width // 2, height // 2))
            cv2_imshow(preview)

    # Cleanup
    cap.release()
    if output_video:
        out.release()
    pbar.close()

    # Calculate processing time
    processing_time = time.time() - processing_start
    actual_fps = frame_count / processing_time

    print(f"\n✅ PROCESSING COMPLETE!")
    print(f"⏱️  Total time: {processing_time:.1f} seconds")
    print(f"📊 Average FPS: {actual_fps:.1f}")
    print(f"👥 Total persons detected: {next_person_id - 1}")

    if output_video:
        print(f"🎬 Output video: {output_path}")

    return analyzer, all_metrics, output_path if output_video else None


In [ ]:
# @title **STEP 6: Run Analysis on Your Video**
print("🎯 READY TO ANALYZE YOUR EXERCISE VIDEO")
print("=" * 60)

# Run the analysis
analyzer, metrics, output_video_path = analyze_exercise_video(video_path)

🎯 READY TO ANALYZE YOUR EXERCISE VIDEO

🚀 STARTING FITNESS ANALYSIS
Input: uploaded_videos/Treadmill.mp4
📦 Loading YOLOv8 pose estimation model...


Processing video:   0%|          | 0/288 [00:00<?, ?it/s]


▶️ Processing frames...


Processing video:   1%|▏         | 4/288 [00:00<00:30,  9.26it/s]

👤 Registered Person 1 (Height: 615 pixels)


Processing video:  69%|██████▉   | 200/288 [00:05<00:02, 31.90it/s, FPS=36.1, Persons=1]

Processing video:  99%|█████████▉| 286/288 [00:08<00:00, 34.73it/s, FPS=36.1, Persons=1]


✅ PROCESSING COMPLETE!
⏱️  Total time: 8.2 seconds
📊 Average FPS: 34.7
👥 Total persons detected: 1
🎬 Output video: output_results/analyzed_Treadmill.mp4


In [ ]:
# @title **STEP 7: Generate Detailed Report**
import pandas as pd
from datetime import datetime

print("\n📋 GENERATING DETAILED FITNESS REPORT")
print("=" * 60)

# Convert metrics to DataFrame
df_metrics = pd.DataFrame(metrics)

if not df_metrics.empty:
    # Create summary for each person
    summaries = []
    for person_id in df_metrics['person_id'].unique():
        summary = analyzer.get_summary(person_id)
        if summary:
            summaries.append(summary)

    # Create summary DataFrame
    df_summary = pd.DataFrame(summaries)

    # Save to CSV
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = f"output_results/fitness_report_{timestamp}.csv"
    df_summary.to_csv(csv_path, index=False)

    print(f"✅ Report saved to: {csv_path}")

    # Display summary
    print("\n" + "=" * 60)
    print("🏃 FITNESS ANALYSIS SUMMARY")
    print("=" * 60)

    for idx, row in df_summary.iterrows():
        print(f"\n👤 PERSON {int(row['person_id'])}:")
        print(f"   Activity: {row['primary_activity']}")
        print(f"   Duration: {row['duration_seconds']:.0f} seconds")
        print(f"   Avg Speed: {row['avg_speed_kmh']:.1f} km/h")
        print(f"   Intensity: {row['intensity_level']}")
        print(f"   Stride Count: {row['stride_count']}")
        if row['avg_cadence_spm'] > 0:
            print(f"   Avg Cadence: {row['avg_cadence_spm']:.0f} steps/min")
        if row['estimated_distance_km'] > 0:
            print(f"   Distance: {row['estimated_distance_km']:.2f} km")
        if row['estimated_calories'] > 0:
            print(f"   Calories: {row['estimated_calories']:.0f} kcal")

    # Overall statistics
    print("\n" + "=" * 60)
    print("📊 OVERALL STATISTICS")
    print("=" * 60)
    print(f"Total persons analyzed: {len(df_summary)}")
    print(f"Total processing time: {len(df_metrics)/analyzer.fps:.1f} seconds")

    if 'speed_kmh' in df_metrics.columns:
        print(f"Overall avg speed: {df_metrics['speed_kmh'].mean():.1f} km/h")
        print(f"Speed range: {df_metrics['speed_kmh'].min():.1f} - {df_metrics['speed_kmh'].max():.1f} km/h")

    # Activity distribution
    if 'activity' in df_metrics.columns:
        activity_counts = df_metrics['activity'].value_counts()
        print("\nActivity Distribution:")
        for activity, count in activity_counts.items():
            percentage = (count / len(df_metrics)) * 100
            print(f"  {activity}: {percentage:.1f}%")
else:
    print("⚠️ No metrics collected. Please check if people were detected in the video.")


📋 GENERATING DETAILED FITNESS REPORT
✅ Report saved to: output_results/fitness_report_20251203_110829.csv

🏃 FITNESS ANALYSIS SUMMARY

👤 PERSON 1:
   Activity: Jogging
   Duration: 2 seconds
   Avg Speed: 10.8 km/h
   Intensity: Vigorous
   Stride Count: 0
   Distance: 0.00 km
   Calories: 0 kcal

📊 OVERALL STATISTICS
Total persons analyzed: 1
Total processing time: 11.4 seconds
Overall avg speed: 6.2 km/h
Speed range: 0.0 - 14.7 km/h

Activity Distribution:
  Walking: 47.9%
  Brisk Walking: 21.5%
  Jogging: 18.3%
  Stationary/Stretching: 12.0%
  Running: 0.4%


In [ ]:
# @title **STEP 8: Visualize Results**
import matplotlib.pyplot as plt

if not df_metrics.empty:
    print("\n📈 GENERATING VISUALIZATIONS")

    # Set up plots
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # 1. Speed over time for each person
    ax1 = axes[0, 0]
    for person_id in df_metrics['person_id'].unique()[:3]:  # First 3 persons
        person_data = df_metrics[df_metrics['person_id'] == person_id]
        ax1.plot(person_data['frame'], person_data['speed_kmh'],
                label=f'Person {person_id}', alpha=0.7)
    ax1.set_title('Speed Over Time')
    ax1.set_xlabel('Frame Number')
    ax1.set_ylabel('Speed (km/h)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # 2. Activity distribution
    ax2 = axes[0, 1]
    if 'activity' in df_metrics.columns:
        activity_counts = df_metrics['activity'].value_counts()
        ax2.bar(activity_counts.index, activity_counts.values)
        ax2.set_title('Activity Distribution')
        ax2.set_xlabel('Activity')
        ax2.set_ylabel('Count')
        ax2.tick_params(axis='x', rotation=45)

    # 3. Intensity levels
    ax3 = axes[1, 0]
    if 'intensity' in df_metrics.columns:
        intensity_counts = df_metrics['intensity'].value_counts()
        colors = ['green', 'lightgreen', 'yellow', 'orange', 'red']
        ax3.pie(intensity_counts.values, labels=intensity_counts.index,
               colors=colors[:len(intensity_counts)], autopct='%1.1f%%')
        ax3.set_title('Intensity Level Distribution')

    # 4. Speed histogram
    ax4 = axes[1, 1]
    if 'speed_kmh' in df_metrics.columns:
        ax4.hist(df_metrics['speed_kmh'], bins=20, edgecolor='black', alpha=0.7)
        ax4.set_title('Speed Distribution')
        ax4.set_xlabel('Speed (km/h)')
        ax4.set_ylabel('Frequency')
        ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("✅ Visualizations generated!")


📈 GENERATING VISUALIZATIONS


✅ Visualizations generated!


In [ ]:
# @title **STEP 9: Download Your Results**
from google.colab import files
import os

print("\n💾 DOWNLOAD YOUR RESULTS")
print("=" * 60)

# List all output files
output_files = []
if os.path.exists('output_results'):
    for file in os.listdir('output_results'):
        if file.endswith('.mp4') or file.endswith('.csv'):
            filepath = f"output_results/{file}"
            filesize = os.path.getsize(filepath) / (1024*1024)  # MB
            output_files.append((filepath, filesize))
            print(f"📁 {file} ({filesize:.1f} MB)")

if output_files:
    print("\nTo download all results, run:")
    print("```python")
    print("# Download the analyzed video:")
    print(f"files.download('{output_files[0][0]}')")
    print()
    print("# Download the CSV report:")
    if len(output_files) > 1:
        print(f"files.download('{output_files[1][0]}')")
    print("```")

    # Create a zip file with all results
    !zip -r /tmp/fitness_results.zip output_results/ 2>/dev/null
    print("\nOr download everything as a ZIP file:")
    print("```python")
    print("files.download('/tmp/fitness_results.zip')")
    print("```")
else:
    print("⚠️ No output files found.")

print("\n" + "=" * 60)
print("✅ ANALYSIS COMPLETE!")
print("=" * 60)
print("\nWhat you've obtained:")
print("1. ✅ Processed video with speed overlays")
print("2. ✅ Detailed CSV report with all metrics")
print("3. ✅ Activity classification (Walking/Jogging/Running)")
print("4. ✅ Intensity levels")
print("5. ✅ Estimated calories and distance")
print("6. ✅ Visualizations and statistics")


💾 DOWNLOAD YOUR RESULTS
📁 fitness_report_20251203_110440.csv (0.0 MB)
📁 analyzed_Treadmill.mp4 (11.5 MB)
📁 analyzed_Running.mp4 (51.3 MB)
📁 fitness_report_20251203_110829.csv (0.0 MB)

To download all results, run:
```python
# Download the analyzed video:
files.download('output_results/fitness_report_20251203_110440.csv')

# Download the CSV report:
files.download('output_results/analyzed_Treadmill.mp4')
```
updating: output_results/ (stored 0%)
updating: output_results/fitness_report_20251203_110440.csv (deflated 52%)
updating: output_results/analyzed_Running.mp4 (deflated 6%)
  adding: output_results/analyzed_Treadmill.mp4 (deflated 5%)
  adding: output_results/fitness_report_20251203_110829.csv (deflated 30%)

Or download everything as a ZIP file:
```python
files.download('/tmp/fitness_results.zip')
```

✅ ANALYSIS COMPLETE!

What you've obtained:
1. ✅ Processed video with speed overlays
2. ✅ Detailed CSV report with all metrics
3. ✅ Activity classification (Walking/Jogging/Running